# Python Tutorial

## Bytes and Bytearray

The core built-in types for manipulating binary data are [`bytes`](https://docs.python.org/3/library/stdtypes.html#bytes) (__immutable__) and [`bytearray`](https://docs.python.org/3/library/stdtypes.html#bytearray) (__mutable__).

Both are sequences of integers in the range 0–255 (one byte each). When displayed, Python renders each byte either as its printable ASCII character (for values 32–126) or as a `\xNN` hex escape for non-printable values (this is purely a display convention, not a property of the data).

A `bytes` literal is written as a string preceded by `b`:

```python
'hello'   # str   — a sequence of Unicode characters
b'hello'  # bytes — a sequence of bytes: 0x68 0x65 0x6c 0x6c 0x6f
```

The key difference is what the two types represent:
- A **string** represents text. The same characters can be encoded in different ways depending on the **encoding** (e.g. [ASCII](https://en.wikipedia.org/wiki/ASCII), [UTF-8](https://en.wikipedia.org/wiki/UTF-8), [UTF-16](https://en.wikipedia.org/wiki/UTF-16)). Encoding is the rule that maps characters to bytes.
- A **bytes object** represents raw binary data — a fixed sequence of byte values with no implicit text meaning attached.

You convert between the two with `encode` and `decode`:

```python
s  = 'hello'
b  = s.encode('utf-8')   # str   -> bytes : b'hello'
s2 = b.decode('utf-8')   # bytes -> str   : 'hello'
```

In cryptography, inputs and outputs are almost always `bytes` (not strings) because encryption operates on raw byte values, not on characters.

In [ ]:
str_ = 'hello'    # a word 'hello'
bytes_ = b'hello' # an immutable sequence of bytes 0x68, 0x65, 0x6c, 0x6c, 0x6f

for char, byte in zip(str_, bytes_):
    print(char, byte, hex(byte))

Like strings, bytes are immutable (they do not support item assignment)

```python
> bytes_[0] = 100
---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[2], line 1
----> 1 bytes_[0] = 100

TypeError: 'bytes' object does not support item assignment
```

Bytearray objects are the mutable counterpart of bytes objects.

In [ ]:
bytearray_ = bytearray(b'hello')
bytearray_[0] = 98
bytearray_

In addition to the literal forms, bytes objects can be created from an iterable of integers.

In [ ]:
# use built-in method `bytes`
bytes_ = bytes([104, 101, 108, 108, 111])
bytes_

Not all integers correspond to a printable ASCII character. When a byte in a bytes object assumes a non-printable value it is diplayed with its hexadecimal representation preceeded by `\x` (e.g., `24` -> `0x18` -> `b'\x18'`)

In [ ]:
bytes_ = bytes([
    0,   # non-printable value
    33,  # value printable as a !
    128, # non-printable value
    52,  # value printable as a 4
    95,  # value printable as a _
    71,  # value printable as a G
    24,  # non-printable value
    125, # value printable as a }
    220, # non-printable value
    98,  # value printable as a b
])
bytes_

A bytes object can be defined from an integer value. However, this definition is not unique and depends on the **endianness**.

Endianness (or "byte-order") describe how computers organize bytes that make up numbers. 
- **big-endian** (BE) means storing the most significant byte at the smallest memory address and the least significant byte at the largest. This approach seems more natural as it is similar to the way digits of numbers are written left-to-right in English.
- **little-endian** (LE) is the opposite order: least significant byte at the smallest memory address and the most significant byte at the largest. This approach is similar to a common European way of writing dates (e.g., 26 March 2024).

Both approaches are widely used. Big-endianness is common in networking procols (such as the Internet) while little-endianness is typical in processor architectures (such as x86, most ARM, and RISC-V).

Let us consider the number `3294399419` that has the hexadecimal representation `0xc45c8bbb`
```
              big-endian        little-endian

             c4 5c 8b bb        c4 5c 8b bb            
    +----+   |  |  |  |         |  |  |  |   +----+    
  0 | c4 | <-+  |  |  |         |  |  |  +-> | bb | 0  
    +----+      |  |  |         |  |  |      +----+    
  1 | 5c | <----+  |  |         |  |  +----> | 8b | 1  
    +----+         |  |         |  |         +----+    
  2 | 8b | <-------+  |         |  +-------> | 5c | 2  
    +----+            |         |            +----+    
  3 | bb | <----------+         +----------> | c4 | 3  
    +----+                                   +----+    
```

In [ ]:
bytes_big = (0x68656c6c6f).to_bytes(5, byteorder='big')
bytes_big

In [ ]:
bytes_little = (0x68656c6c6f).to_bytes(5, byteorder='little')
bytes_little

**Exercise**
1. Create a `bytes` object of 1000 random bytes (use the function `randint` from `random` built-in package)
2. Store the `bytes` object in a binary file named 'random-bytes.bin'.
3. Try to load the same file as a text file.
4. Create a new bytes object of 1000 random bytes. However, this time limits the range of possible values from 32 to 127, included.
5. Store the new `bytes` object in a binary file named 'random-bytes.bin'.
6. Try to load the same file as a text file.

**Exercise**
1. Generate a random number of 32 bits.
2. Transform the generated number into a `bytes` object (big-endian)
3. Transform the `bytes` object into a sequence (list) of boolean (big-endian)
4. Compute the parity bit

## Bits

Working at the level of individual bits is not directly supported by Python's built-in types — `bytes` and `bytearray` operate on whole bytes (8 bits at a time). For this course we provide a custom class `Bits`, defined in [`src/bits.py`](../src/bits.py), that represents an ordered sequence of bits and supports the operations most relevant to cryptography.

`Bits` is **not** a black box: you are expected to read and modify it. If you find a missing operation or a behaviour that does not suit a particular exercise, open `src/bits.py` and change it.

A `Bits` object can be constructed from several input types:

| Input | Example | Meaning |
|---|---|---|
| `list` / `tuple` of `bool` or `int` | `Bits([True, False, True])` | direct bit sequence |
| `int` | `Bits(42)` | binary representation of the integer |
| `bytes` | `Bits(b'\xf0')` | bit expansion of each byte |
| `str` of `'0'`/`'1'` | `Bits("1010")` | parse a binary string |
| another `Bits` | `Bits(other)` | copy |

The class also provides conversions back to `int`, `bytes`, and sparse index sets, as well as the bitwise operators `^`, `&`, `|`, `~`, concatenation `+`, and replication `*`.

In [ ]:
from bits import Bits

In [ ]:
a = Bits([False, True, False, False, False, True, True, False])
print(a.bits)
# [False, True, False, False, False, True, True, False]

In [ ]:
print(Bits(31), Bits(b'\xf0'), Bits([True, True, False]))  # 11111 11110000 110

In [ ]:
a = Bits(76)
print(a.bits)  # [True, False, False, True, True, False, False]

In [ ]:
a = Bits(b'$[')
a.append(False)
print(a)  # 00100100010110110

In [ ]:
a = 33264
b = b'\xfa\xe3'
print(Bits(a))            # 1000000111110000
print(Bits(b))            # 1111101011100011
print(Bits(a) ^ Bits(b))  # 0111101100010011

In [ ]:
a = 33264
b = b'\xfa\xe3'
print(Bits(a))            # 1000000111110000
print(Bits(b))            # 1111101011100011
print(Bits(a) & Bits(b))  # 1000000011100000

In [ ]:
a = 33264
b = b'\xfa\xe3'
print(Bits(a), Bits(b))   # 1000000111110000 1111101011100011
print(Bits(a) + Bits(b))  # 10000001111100001111101011100011

In [ ]:
Bits(33264).to_bytes()  # b'\x81\xf0'

In [ ]:
a = Bits(b'\x08\xd2')
print(a)  # 0000100011010010
a.append(True)
print(a)  # 00001000110100101

In [ ]:
a = Bits(b'\x08\xd2')
print(a)  # 0000100011010010
a.pop(4)
print(a)  # 000000011010010